In [7]:
import pandas as pd
import numpy as np
from scipy.stats import t
from scipy.stats import ttest_ind_from_stats

# =====================================
# CONFIGURAÇÕES
# =====================================

CSV_FILE = "melhores_resultados.csv"

N_FOLDS = 5
ALPHA = 0.05
CONFIDENCE = 0.95

# =====================================
# FUNÇÃO IC95%
# =====================================

def calcular_ic(media, std, n=N_FOLDS):

    t_value = t.ppf((1 + CONFIDENCE) / 2, df=n-1)

    margem = t_value * (std / np.sqrt(n))

    return media - margem, media + margem

# =====================================
# LEITURA DO CSV
# =====================================

print("Lendo arquivo CSV...")

df = pd.read_csv(CSV_FILE)



Lendo arquivo CSV...


In [8]:
# =====================================
# INTERVALOS DE CONFIANÇA
# =====================================

print("Calculando IC95%...")

ic_resultados = []

for _, row in df.iterrows():

    ic_inf, ic_sup = calcular_ic(
        row['f1_score'],
        row['f1_std']
    )

    ic_resultados.append({
        'dataset': row['dataset'],
        'abordagem': row['abordagem'],
        'modelo': row['modelo'],
        'f1_score': row['f1_score'],
        'f1_std': row['f1_std'],
        'ic_inf': round(ic_inf, 4),
        'ic_sup': round(ic_sup, 4)
    })

df_ic = pd.DataFrame(ic_resultados)

# =====================================
# TESTES ESTATÍSTICOS
# =====================================

print("Executando testes estatísticos...")

comparacoes = []

# =====================================
# 1. COMPARAÇÕES INTRA-DATASET
# =====================================

for dataset in df['dataset'].unique():

    print(f"\nProcessando dataset: {dataset}")

    df_ds = df[df['dataset'] == dataset]

    for i in range(len(df_ds)):
        for j in range(i + 1, len(df_ds)):

            a = df_ds.iloc[i]
            b = df_ds.iloc[j]

            stat, p = ttest_ind_from_stats(
                mean1=a['f1_score'],
                std1=a['f1_std'],
                nobs1=N_FOLDS,

                mean2=b['f1_score'],
                std2=b['f1_std'],
                nobs2=N_FOLDS,

                equal_var=False
            )

            melhor = (
                a['abordagem']
                if a['f1_score'] > b['f1_score']
                else b['abordagem']
            )

            comparacoes.append({
                'tipo': 'intra_dataset',
                'dataset': dataset,

                'abordagem_1': a['abordagem'],
                'abordagem_2': b['abordagem'],

                'f1_1': round(a['f1_score'], 4),
                'f1_2': round(b['f1_score'], 4),

                'p_value': round(p, 6),

                'significativo':
                    'Sim' if p < ALPHA else 'Não',

                'melhor_abordagem': melhor
            })



Calculando IC95%...
Executando testes estatísticos...

Processando dataset: PrejudiceWhatsApp-br

Processando dataset: PrejudiceTelegram-br


In [9]:
# =====================================
# 2. COMPARAÇÕES INTER-DATASET
# =====================================

print("\nComparando datasets...")

abordagens = df['abordagem'].unique()

for abordagem in abordagens:

    subset = df[df['abordagem'] == abordagem]

    wpp = subset[
        subset['dataset'] == 'PrejudiceWhatsApp-br'
    ]

    tel = subset[
        subset['dataset'] == 'PrejudiceTelegram-br'
    ]

    if len(wpp) == 0 or len(tel) == 0:
        continue

    a = wpp.iloc[0]
    b = tel.iloc[0]

    stat, p = ttest_ind_from_stats(
        mean1=a['f1_score'],
        std1=a['f1_std'],
        nobs1=N_FOLDS,

        mean2=b['f1_score'],
        std2=b['f1_std'],
        nobs2=N_FOLDS,

        equal_var=False
    )

    melhor = (
        a['dataset']
        if a['f1_score'] > b['f1_score']
        else b['dataset']
    )

    comparacoes.append({
        'tipo': 'inter_dataset',

        'dataset': abordagem,

        'abordagem_1': 'PrejudiceWhatsApp-br',
        'abordagem_2': 'PrejudiceTelegram-br',

        'f1_1': round(a['f1_score'], 4),
        'f1_2': round(b['f1_score'], 4),

        'p_value': round(p, 6),

        'significativo':
            'Sim' if p < ALPHA else 'Não',

        'melhor_abordagem': melhor
    })




Comparando datasets...


In [10]:
# =====================================
# DATAFRAME FINAL
# =====================================

comparacoes_df = pd.DataFrame(comparacoes)

# =====================================
# MATRIZ 1 - WHATSAPP
# =====================================

print("\nGerando matriz WhatsApp...")

wpp_df = comparacoes_df[
    (comparacoes_df['tipo'] == 'intra_dataset') &
    (comparacoes_df['dataset'] == 'PrejudiceWhatsApp-br')
]

matriz_wpp = wpp_df.pivot_table(
    index='abordagem_1',
    columns='abordagem_2',
    values='p_value',
    aggfunc='first'
)

# =====================================
# MATRIZ 2 - TELEGRAM
# =====================================

print("Gerando matriz Telegram...")

tel_df = comparacoes_df[
    (comparacoes_df['tipo'] == 'intra_dataset') &
    (comparacoes_df['dataset'] == 'PrejudiceTelegram-br')
]

matriz_tel = tel_df.pivot_table(
    index='abordagem_1',
    columns='abordagem_2',
    values='p_value',
    aggfunc='first'
)

# =====================================
# MATRIZ 3 - INTER DATASETS
# =====================================

print("Gerando matriz inter-datasets...")

inter_df = comparacoes_df[
    comparacoes_df['tipo'] == 'inter_dataset'
]

matriz_inter = inter_df.pivot_table(
    index='dataset',
    columns='melhor_abordagem',
    values='p_value',
    aggfunc='first'
)

# =====================================
# INSIGHTS AUTOMÁTICOS
# =====================================

print("Gerando insights...")

insights = []

# Melhor abordagem por dataset

for dataset in df['dataset'].unique():

    melhor = df[
        df['dataset'] == dataset
    ].sort_values(
        by='f1_score',
        ascending=False
    ).iloc[0]

    insights.append({
        'tipo': 'melhor_modelo',
        'descricao':
            f"No dataset {dataset}, "
            f"a melhor abordagem foi "
            f"'{melhor['abordagem']}' "
            f"utilizando o modelo "
            f"{melhor['modelo']} "
            f"com F1-score de "
            f"{melhor['f1_score']:.3f}."
    })

# Diferenças significativas

sig = comparacoes_df[
    comparacoes_df['significativo'] == 'Sim'
]

for _, row in sig.iterrows():

    insights.append({
        'tipo': 'diferenca_significativa',
        'descricao':
            f"Foi observada diferença "
            f"estatisticamente significativa "
            f"entre '{row['abordagem_1']}' "
            f"e '{row['abordagem_2']}' "
            f"(p = {row['p_value']})."
    })

insights_df = pd.DataFrame(insights)

# =====================================
# SALVAR RESULTADOS
# =====================================

print("\nSalvando arquivos...")

# Intervalos de confiança
df_ic.to_csv(
    'intervalos_confianca.csv',
    index=False
)

# Todas comparações
comparacoes_df.to_csv(
    'comparacoes_estatisticas.csv',
    index=False
)

# Insights
insights_df.to_csv(
    'insights_estatisticos.csv',
    index=False
)

# Matrizes
matriz_wpp.to_csv(
    'matriz_pvalues_whatsapp.csv'
)

matriz_tel.to_csv(
    'matriz_pvalues_telegram.csv'
)

matriz_inter.to_csv(
    'matriz_pvalues_inter_datasets.csv'
)

print("\nArquivos gerados com sucesso:")
print("- intervalos_confianca.csv")
print("- comparacoes_estatisticas.csv")
print("- insights_estatisticos.csv")
print("- matriz_pvalues_whatsapp.csv")
print("- matriz_pvalues_telegram.csv")
print("- matriz_pvalues_inter_datasets.csv")


Gerando matriz WhatsApp...
Gerando matriz Telegram...
Gerando matriz inter-datasets...
Gerando insights...

Salvando arquivos...

Arquivos gerados com sucesso:
- intervalos_confianca.csv
- comparacoes_estatisticas.csv
- insights_estatisticos.csv
- matriz_pvalues_whatsapp.csv
- matriz_pvalues_telegram.csv
- matriz_pvalues_inter_datasets.csv
